# PitchLens — initial dataset exploration

This notebook is the entry point for understanding the data before transforming it or training a model. It answers four questions:

1. Which sources do we have, and under what conditions can we use them?
2. Are the annotations technically valid?
3. How large will the ball actually be after resizing to 640 px?
4. Can we trust the existing train/validation/test splits?

The cells call the tested functions in `src/pitchlens/data`. The notebook keeps the explanations, the charts and the decisions. It never modifies the raw data.

## 0. Setup

The next cell locates the project root, even when the notebook is opened from the `notebooks/` folder, then makes the local package importable.

In [ ]:
import json
import sys
import tomllib
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from PIL import Image, ImageDraw

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from pitchlens.data.audit_roboflow_splits import audit_splits, parent_name
from pitchlens.data.audit_sequences import audit_sequences
from pitchlens.data.audit_yolo import audit, label_path_for

INFACTORY_ROOT = PROJECT_ROOT / "data/raw/infactory_ball_tracking/data"
SOCCEREYE_ROOT = PROJECT_ROOT / "data/interim/soccereye"

print(f"Projet : {PROJECT_ROOT}")
print(f"Infactory disponible : {INFACTORY_ROOT.is_dir()}")
print(f"SoccerEye disponible : {SOCCEREYE_ROOT.is_dir()}")

## 1. Source inventory

The registry does not contain the images. It records their origin, their license and their status in the project. A non-commercial license is acceptable for the current experiments; a missing license remains an open point to clarify before any republication.

In [ ]:
registry_path = PROJECT_ROOT / "configs/data_sources.toml"
with registry_path.open("rb") as stream:
    sources = tomllib.load(stream)["sources"]

rows = ["| Source | Status | License | Domain |", "|---|---|---|---|"]
for source in sources:
    rows.append(
        f"| {source['name']} | `{source['status']}` | "
        f"{source['data_license']} | {source['domain']} |"
    )
display(Markdown("\n".join(rows)))

### Origin of the two datasets available locally

A dataset is more than its images: we need to be able to trace its producer, the original page, the exact version and the terms of use. The information below comes from the manifests recorded at download time.

In [ ]:
manifest_directory = PROJECT_ROOT / "data/manifests"
infactory_manifest = json.loads(
    (manifest_directory / "infactory_ball_tracking.json").read_text()
)
soccereye_manifest = json.loads(
    (manifest_directory / "soccereye.json").read_text()
)

display(Markdown(f"""
| | Infactory Ball Tracking | SoccerEye |
|---|---|---|
| Producer | Infactory AI | WWandP / SoccerEye repository |
| Original page | [Hugging Face]({infactory_manifest['source_url']}) | [GitHub](https://github.com/WWandP/SoccerEye) then Google Drive |
| View type | Professional broadcasts, wide shots and close-ups | Professional/scouting views, sometimes tiled |
| Annotation | One class: ball; frames without a TXT file = negatives | Two classes: ball and person |
| Local version | Commit `{infactory_manifest['resolved_revision'][:12]}` | SHA-256 archive `{soccereye_manifest['archive_sha256'][:12]}…` |
| License used | CC BY-NC 4.0 | Data license not specified |
| PitchLens role | Initial temporal base | Complement for tiny balls |
"""))

print("Infactory raw data:", INFACTORY_ROOT)
print("SoccerEye extracted data:", SOCCEREYE_ROOT)

## 2. Infactory: actual content

`audit_yolo` checks every YOLO line, opens the images to get their resolution and converts the normalized boxes to pixels. `target_size=640` simulates a proportional resize with letterboxing, as in a standard YOLO training run.

In [ ]:
if not INFACTORY_ROOT.is_dir():
    raise FileNotFoundError("Infactory has not been downloaded. See download_hf.py.")

infactory = audit(
    INFACTORY_ROOT,
    missing_label_is_negative=True,
    target_size=640,
)
ball = infactory["class_metrics"][0]
ball_640 = ball["letterbox_640_px"]

display(Markdown(f"""
| Metric | Value |
|---|---:|
| Images | {infactory['images']:,} |
| Images with a valid ball box | {ball['images_with_class']:,} |
| Explicit negative images | {infactory['negative_images']:,} |
| Valid boxes | {ball['boxes']:,} |
| Invalid annotations | {infactory['issues_count']:,} |
| Images with several ball boxes | {ball['multi_box_images_count']:,} |
| Median ball at native resolution | {ball['width_px']['median']:.1f} × {ball['height_px']['median']:.1f} px |
| Median ball after 640 letterbox | {ball_640['width']['median']:.1f} × {ball_640['height']['median']:.1f} px |
"""))

print("Resolutions:", infactory["resolution_counts"])

### Why sequences change how we build the splits

Two neighboring frames are almost identical. They must never end up one in train and the other in test. The unit of separation is therefore the full sequence UUID, not the image.

In [ ]:
infactory_sequences = audit_sequences(INFACTORY_ROOT)
lengths = [row["frames"] for row in infactory_sequences["sequence_details"]]

display(Markdown(f"""
- **{infactory_sequences['sequences']} sequences** for {infactory_sequences['parsed_frames']:,} frames;
- minimum / median / maximum length: **{min(lengths)} / {infactory_sequences['sequence_length_frames']['median']:.0f} / {max(lengths)} frames**;
- **{infactory_sequences['contiguous_sequences']} contiguous sequences**, with no missing frame number;
- **{infactory_sequences['all_negative_sequences']} fully negative sequences**;
- **{infactory_sequences['sequences_with_invalid_annotations']} sequences** containing invalid annotations.
"""))

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(lengths, bins=15, color="#3b82f6", edgecolor="white")
ax.set(title="Infactory sequence length", xlabel="Number of frames", ylabel="Sequences")
plt.show()

### Visual inspection

The red boxes below are read directly from the YOLO files. A normal case, a negative, a multi-box case and an invalid case are displayed automatically. The goal is not to fix anything yet: it is to understand what the statistics hide.

In [ ]:
def valid_boxes(image_path, dataset_root):
    label_path = label_path_for(image_path, dataset_root)
    if not label_path.exists():
        return []
    boxes = []
    with Image.open(image_path) as image:
        width, height = image.size
    for row in label_path.read_text().splitlines():
        fields = row.split()
        if len(fields) != 5:
            continue
        try:
            class_id = int(fields[0])
            x, y, box_width, box_height = map(float, fields[1:])
        except ValueError:
            continue
        if box_width <= 0 or box_height <= 0:
            continue
        boxes.append((
            class_id,
            (x - box_width / 2) * width,
            (y - box_height / 2) * height,
            (x + box_width / 2) * width,
            (y + box_height / 2) * height,
        ))
    return boxes

def annotated_image(image_path, dataset_root, class_ids=None):
    image = Image.open(image_path).convert("RGB")
    drawing = ImageDraw.Draw(image)
    for class_id, left, top, right, bottom in valid_boxes(image_path, dataset_root):
        if class_ids is not None and class_id not in class_ids:
            continue
        color = "red" if class_id == 0 else "deepskyblue"
        line_width = 4 if class_id == 0 else 2
        drawing.rectangle((left, top, right, bottom), outline=color, width=line_width)
    return image

positive = next(
    path for path in sorted((INFACTORY_ROOT / "visible").glob("*.jpg"))
    if len(valid_boxes(path, INFACTORY_ROOT)) == 1
)
negative = next((INFACTORY_ROOT / "not_visible").glob("*.jpg"))
multi = INFACTORY_ROOT / ball["multi_box_image_examples"][0]
invalid_label = infactory["issues"][0].split(": ", 1)[1].rsplit(":", 1)[0]
invalid = (INFACTORY_ROOT / invalid_label).with_suffix(".jpg")

examples = [
    ("One valid box", positive),
    ("Negative: no ball annotated", negative),
    ("Several boxes: review needed", multi),
    ("Zero-area box", invalid),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for axis, (title, image_path) in zip(axes.flat, examples):
    axis.imshow(annotated_image(image_path, INFACTORY_ROOT))
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()
plt.show()

### Infactory: from the smallest to the largest ball

A median alone hides very different situations. This gallery automatically selects six images according to the area of their box: minimum, 10th percentile, median, 90th percentile and maximum. It shows in particular that the very large boxes often come from TV close-ups, which are not representative of a fixed amateur camera.

In [ ]:
infactory_single_boxes = []
for image_path in sorted((INFACTORY_ROOT / "visible").glob("*.jpg")):
    boxes = valid_boxes(image_path, INFACTORY_ROOT)
    if len(boxes) != 1:
        continue
    _, left, top, right, bottom = boxes[0]
    area = (right - left) * (bottom - top)
    infactory_single_boxes.append((area, image_path, right - left, bottom - top))

infactory_single_boxes.sort(key=lambda row: row[0])
quantiles = [0.00, 0.10, 0.25, 0.50, 0.90, 1.00]
selected = [
    infactory_single_boxes[round((len(infactory_single_boxes) - 1) * quantile)]
    for quantile in quantiles
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for axis, quantile, (_, image_path, width, height) in zip(axes.flat, quantiles, selected):
    axis.imshow(annotated_image(image_path, INFACTORY_ROOT, class_ids={0}))
    axis.set_title(f"quantile {quantile:.0%} — {width:.1f} × {height:.1f} px")
    axis.axis("off")
plt.tight_layout()
plt.show()

### Infactory: one sequence over time

The following images all come from the same UUID. `visible` means a valid box exists; `negative` means the ball is absent, hidden or out of frame. This view explains why we want to keep sequences for tracking, but also why an entire sequence must stay in a single split.

In [ ]:
mixed_sequences = [
    row for row in infactory_sequences["sequence_details"]
    if row["positive_frames"] > 0 and row["negative_frames"] > 0
]
sequence = max(mixed_sequences, key=lambda row: row["frames"])
sequence_id = sequence["sequence"]
frame_numbers = [
    round(index * sequence["last_frame"] / 7) for index in range(8)
]

sequence_samples = []
for frame_number in frame_numbers:
    filename = f"{sequence_id}_{frame_number:06d}.jpg"
    visible_path = INFACTORY_ROOT / "visible" / filename
    negative_path = INFACTORY_ROOT / "not_visible" / filename
    image_path = visible_path if visible_path.exists() else negative_path
    state = "visible" if visible_path.exists() else "negative"
    sequence_samples.append((frame_number, state, image_path))

fig, axes = plt.subplots(2, 4, figsize=(17, 8))
for axis, (frame_number, state, image_path) in zip(axes.flat, sequence_samples):
    axis.imshow(annotated_image(image_path, INFACTORY_ROOT, class_ids={0}))
    axis.set_title(f"frame {frame_number} — {state}")
    axis.axis("off")
fig.suptitle(f"Sequence {sequence_id[:8]}… ({sequence['frames']} frames)", fontsize=14)
plt.tight_layout()
plt.show()

## 3. SoccerEye: value and limits

SoccerEye has two classes: `0 = ball`, `1 = person`. For PitchLens, an image that only contains people is a valid negative for the ball class. The leakage audit also analyzes the Roboflow names and the SHA-256 fingerprints.

In [ ]:
if not SOCCEREYE_ROOT.is_dir():
    raise FileNotFoundError("SoccerEye has not been extracted into data/interim.")

soccereye = audit(SOCCEREYE_ROOT, target_size=640)
soccereye_leakage = audit_splits(SOCCEREYE_ROOT)
soccer_ball = soccereye["class_metrics"][0]
soccer_ball_640 = soccer_ball["letterbox_640_px"]

display(Markdown(f"""
| Metric | Value |
|---|---:|
| Images | {soccereye['images']:,} |
| Images with a ball | {soccer_ball['images_with_class']:,} |
| Images without a ball | {soccer_ball['images_without_class']:,} |
| Ball boxes | {soccer_ball['boxes']:,} |
| Median ball after 640 letterbox | {soccer_ball_640['width']['median']:.1f} × {soccer_ball_640['height']['median']:.1f} px |
| Parent image names present in several splits | {soccereye_leakage['parent_names_across_splits']:,} |
| Source prefixes present in several splits | {soccereye_leakage['source_prefixes_across_splits']:,} / {soccereye_leakage['unique_source_prefixes']:,} |
| Strictly identical files across splits | {soccereye_leakage['exact_file_hashes_across_splits']:,} |
"""))

### SoccerEye: inspecting the annotated balls

The person class is deliberately hidden here to keep the ball readable. The six examples are chosen from the smallest to the largest ball. The title also shows the split and the original resolution.

In [ ]:
soccer_images = sorted(
    path for path in SOCCEREYE_ROOT.rglob("*.jpg") if "images" in path.parts
)
soccer_ball_images = []
for image_path in soccer_images:
    ball_boxes = [box for box in valid_boxes(image_path, SOCCEREYE_ROOT) if box[0] == 0]
    if not ball_boxes:
        continue
    largest_ball = max(
        ball_boxes,
        key=lambda box: (box[3] - box[1]) * (box[4] - box[2]),
    )
    _, left, top, right, bottom = largest_ball
    soccer_ball_images.append(
        ((right - left) * (bottom - top), image_path, right - left, bottom - top)
    )

soccer_ball_images.sort(key=lambda row: row[0])
quantiles = [0.00, 0.10, 0.25, 0.50, 0.90, 1.00]
selected = [
    soccer_ball_images[round((len(soccer_ball_images) - 1) * quantile)]
    for quantile in quantiles
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for axis, quantile, (_, image_path, width, height) in zip(axes.flat, quantiles, selected):
    with Image.open(image_path) as image:
        resolution = f"{image.width}×{image.height}"
    split = next(part for part in image_path.parts if part in {"train", "valid", "test"})
    axis.imshow(annotated_image(image_path, SOCCEREYE_ROOT, class_ids={0}))
    axis.set_title(
        f"q{quantile:.0%} — {split} — {resolution}\nballon {width:.1f} × {height:.1f} px"
    )
    axis.axis("off")
plt.tight_layout()
plt.show()

### SoccerEye: seeing the leakage between splits

Each row below shows two variants that share the same Roboflow parent name but are placed in two different splits. They are not byte-for-byte identical, but they come from the same parent image. An evaluation built this way can reward memorization rather than generalization.

In [ ]:
leakage_pairs = []
for parent in soccereye_leakage["parent_split_examples"]:
    matches = [path for path in soccer_images if parent_name(path) == parent]
    by_split = {}
    for image_path in matches:
        split = next(part for part in image_path.parts if part in {"train", "valid", "test"})
        by_split.setdefault(split, image_path)
    if len(by_split) >= 2:
        first_two = sorted(by_split.items())[:2]
        leakage_pairs.append((parent, first_two))
    if len(leakage_pairs) == 3:
        break

fig, axes = plt.subplots(len(leakage_pairs), 2, figsize=(13, 5 * len(leakage_pairs)))
for row_axes, (parent, pair) in zip(axes, leakage_pairs):
    for axis, (split, image_path) in zip(row_axes, pair):
        axis.imshow(annotated_image(image_path, SOCCEREYE_ROOT, class_ids={0}))
        axis.set_title(f"parent {parent} — split {split}")
        axis.axis("off")
plt.tight_layout()
plt.show()

## 4. Comparison with the mobile input size

Comparing native pixels would be misleading since the resolutions differ. The chart therefore only uses the dimensions after letterboxing to `640 × 640`.

In [ ]:
names = ["Infactory", "SoccerEye"]
widths = [ball_640["width"]["median"], soccer_ball_640["width"]["median"]]
heights = [ball_640["height"]["median"], soccer_ball_640["height"]["median"]]
positions = range(len(names))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar([index - 0.18 for index in positions], widths, width=0.36, label="width")
axes[0].bar([index + 0.18 for index in positions], heights, width=0.36, label="height")
axes[0].set_xticks(list(positions), names)
axes[0].set_ylabel("Pixels at 640 input")
axes[0].set_title("Median ball size")
axes[0].legend()

with_ball = [ball["images_with_class"], soccer_ball["images_with_class"]]
without_ball = [
    infactory["images"] - ball["images_with_class"],
    soccer_ball["images_without_class"],
]
axes[1].bar(names, with_ball, label="with valid ball")
axes[1].bar(names, without_ball, bottom=with_ball, label="without valid ball")
axes[1].set_ylabel("Images")
axes[1].set_title("Ball presence")
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Conclusion and decision

### What the data allows today

- **Infactory** is the best choice for building a first baseline: more balls, temporal sequences and true negatives.
- **SoccerEye** is an interesting complement for very small balls, but not a reliable evaluation base with its current splits.
- Both sources mostly come from professional broadcasts. They do not yet prove that a fixed iPhone will track an amateur match correctly.

### Next step

Build `data/processed/v001` from Infactory only:

1. review the 75 invalid annotations;
2. review the 63 multi-box images;
3. create the splits by full UUID;
4. record the composition and statistics of `v001`;
5. then train a first nano baseline.

SoccerEye will be tested separately in a second version. This separation will let us measure its actual contribution instead of assuming that more data necessarily improves the model.